## Bronze layer — data validation

### Setup
- Catalog: `kitsune_project`
- Schema: `bronze_layer`
- Volume: `raw_files`
- Source files: `SYN DoS_dataset.csv.gz` (~ 2 GB), `SYN DoS_labels.csv.gz` (~ 6 MB)

### Validation steps

**1. File existence and size check**
Confirmed both files are present in the volume with sizes matching the
original upload (2,136,972,919 bytes and 6,568,025 bytes).

**2. Schema check (no header assumption)**
Read the dataset file with `header=False`, `inferSchema=False`.
Confirmed 115 columns, matching the paper's documented feature count.
First rows contain only numeric values — no header row present in the
dataset file.

**3. Row count validation**
| File | Row count |
|---|---|
| `SYN DoS_dataset.csv.gz` | 2,771,276 |
| `SYN DoS_labels.csv.gz` | 2,771,277 |

**Mismatch found: labels file has exactly 1 extra row.**

In [0]:
# Data ingestion setup
# Point this Spark session to the correct catalog and schema in Unity Catalog
spark.sql("USE CATALOG kitsune_project")
spark.sql("USE SCHEMA bronze_layer")

# List the raw files inside the volume to confirm they exist
# and that the upload sizes match what we expect
files = dbutils.fs.ls("/Volumes/kitsune_project/bronze_layer/raw_files/")
for f in files:
    print(f"{f.name} — {f.size} bytes")

In [0]:
# Spark automatically decompresses .gz files while reading, no manual unzip needed
# inferSchema=False avoids forcing Spark to scan the full 2GB file just to guess types
df_check = spark.read.csv(
    "/Volumes/kitsune_project/bronze_layer/raw_files/SYN DoS_dataset.csv.gz",
    header=False,
    inferSchema=False 
)

print(f"Number of columns: {len(df_check.columns)}")
df_check.show(3)

In [0]:
# Confirm dataset and labels have the same number of rows
# before attempting to join them later — a mismatch here would
# silently misalign features with labels
dataset_count = df_check.count()

df_labels_check = spark.read.csv(
    "/Volumes/kitsune_project/bronze_layer/raw_files/SYN DoS_labels.csv.gz",
    header=False,
    inferSchema=False
)
labels_count = df_labels_check.count()

print(f"Dataset rows: {dataset_count}")
print(f"Labels rows: {labels_count}")
print(f"Match: {dataset_count == labels_count}")

The 1-row difference suggests the labels file likely contains a header
row (e.g. `"x"` or `"label"`) while the dataset file does not. Both files
were read with `header=False`, so if this hypothesis is correct, the
header text was ingested as if it were a data row in the labels file.

In [0]:
# Check whether the first row of labels looks like a header (text) or real data (0/1)
df_labels_check.show(3)

## Labels file has a header

Row 1 of the labels file was actually column names, not real data
(`"x"` = 0 or 1). That's why the row counts didn't match earlier.

Also found: the labels file has 2 columns, not 1 — a row number we
don't need, and the real label. Fixing this by re-reading with
`header=True` and dropping the row number column.

In [0]:
# Re-read labels with header=True now that we confirmed row 1 contains column names
# The first column is just a row index (not useful), the second is the actual label
df_labels_check = spark.read.csv(
    "/Volumes/kitsune_project/bronze_layer/raw_files/SYN DoS_labels.csv.gz",
    header=True,
    inferSchema=False
)

df_labels_check.show(3)
print(f"Columns: {df_labels_check.columns}")
print(f"Row count: {df_labels_check.count()}")

In [0]:
# Rename columns to something meaningful
# _c0 was an unused row index from the original file, we drop it
# x becomes "label" — clearer for anyone reading this code later
df_labels_check = df_labels_check.drop("_c0").withColumnRenamed("x", "label")

df_labels_check.show(3)

# Rename generic column names to something more descriptive
new_names = [f"feature_{i+1}" for i in range(len(df_check.columns))]
df_check = df_check.toDF(*new_names)

df_check.printSchema()

In [0]:
from pyspark.sql import functions as F

features_partitions = df_check.select(F.spark_partition_id()).distinct().count()
labels_partitions = df_labels_check.select(F.spark_partition_id()).distinct().count()

print(f"Features partitions: {features_partitions}")
print(f"Labels partitions: {labels_partitions}")

## Row_id alignment verification

Before joining `df_check` (features) and `df_labels_check` (labels) on `row_id`,
verified that `monotonically_increasing_id()` would produce correctly aligned IDs
across both DataFrames.

**Why this matters:** `monotonically_increasing_id()` only guarantees a strictly
sequential order within a single partition. If either DataFrame had been split
across multiple partitions, the generated IDs could correspond to different rows
in each DataFrame, silently misaligning features with labels — with no error and
no change in row count to signal the problem.

**Result:** 1 partition for both DataFrames.

**Conclusion:** with exactly one partition per DataFrame, `monotonically_increasing_id()`
generated a strictly sequential order (0, 1, 2, ...) matching the row order in each
source file. The join on `row_id` is therefore correctly aligned — no feature/label
mismatch is possible under this partitioning.

## Renamed the columns

**Labels file:** removed a leftover row-count column we didn't need,
and renamed the label column from `x` to something clearer: `label`
(0 = normal, 1 = attack).

**Dataset file:** the 115 columns had no real names (just `_c0`, `_c1`...),
so we gave them simple, consistent names: `feature_1` through
`feature_115`. There's no official list of exact names for what each
one measures, so we kept it simple instead of guessing.

In [0]:
# Add a row number to each DataFrame so we can join them by position
from pyspark.sql.functions import monotonically_increasing_id

df_check = df_check.withColumn("row_id", monotonically_increasing_id())
df_labels_check = df_labels_check.withColumn("row_id", monotonically_increasing_id())

# Join both DataFrames on that shared row_id
df_bronze = df_check.join(df_labels_check, on="row_id", how="inner")

# Confirm the join didn't lose or duplicate any rows
print(f"Joined row count: {df_bronze.count()}")
df_bronze.select("row_id", "feature_1", "feature_2", "label").show(3)

In [0]:
# Save the joined data as a Delta table inside the bronze schema
df_bronze.write.format("delta").mode("overwrite").saveAsTable(
    "kitsune_project.bronze_layer.syn_dos_raw"
)

print("Table saved successfully")

In [0]:
# Read the saved Delta table back as a DataFrame
df_kitsune_combined = spark.read.table("kitsune_project.bronze_layer.syn_dos_raw")

# Display row count and column count
print(f"Number of rows: {df_kitsune_combined.count()}")
print(f"Number of columns: {len(df_kitsune_combined.columns)}")

# Show the first few rows
df_kitsune_combined.show(5)

## Summary — what we did in this notebook

We took two raw files from the SYN DoS attack (part of the Kitsune
network intrusion dataset) and turned them into one clean, reliable
table saved permanently in Databricks.

**What we started with:**
- A file with 115 columns of network traffic measurements (~2 GB)
- A separate file saying whether each row was normal traffic (0) or
  an attack (1)

**What we found and fixed along the way:**
- The two files didn't line up — the labels file had one extra row.
  Turned out it had a header row we weren't expecting.
- The labels file also had an unused row-count column we didn't need.
- None of the 115 feature columns had real names, so we gave them
  simple, consistent ones (`feature_1` to `feature_115`).
- Verified that the row_id-based join between features and labels was safe:
  both source files were read into a single Spark partition each (gzip files
  aren't splittable), which guarantees `monotonically_increasing_id()` produced
  a correctly aligned, sequential order in both DataFrames.

**What we ended up with:**
- Both files matched perfectly after the fix: 2,771,276 rows each
- Combined them into a single table, `syn_dos_raw`, with all features
  plus the label in one place
- Saved it as a permanent Delta table inside Databricks, so it won't
  disappear if the session restarts

**Next notebook:**
- All 115 feature columns are still stored as text, not numbers —
  that gets fixed in the next step (Silver layer)